In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage,BaseMessage,AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, MessagesState, START, END
from dotenv import load_dotenv
from typing import Literal,Annotated,TypedDict
from pydantic import BaseModel,Field
import operator

load_dotenv()

True

In [26]:
class ChatState(TypedDict):
    chat_history: Annotated[list[BaseMessage], operator.add]

In [33]:
llm=ChatGoogleGenerativeAI(model='gemini-3.7-flash')

In [30]:
def chat_node(state: ChatState):
    user_input = input("ASK ANYTHING FROM AI: ")

    if user_input.strip().lower() in ["exit", "bye"]:
        return {
            "chat_history": [
                HumanMessage(content=user_input)
            ]
        }

    response = llm.invoke(user_input)

    return {
        "chat_history": [
            HumanMessage(content=user_input),
            AIMessage(content=response.content)
        ]
    }


def checkExit(state: ChatState):
    messages = state["chat_history"]

    last_message = messages[-1]

    if last_message.content.strip().lower() in ["exit", "bye"]:
        return "Khatam"

    return "chat_node"


def Khatam(state: ChatState):
    return {}

In [31]:
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_node("checkExit", lambda state: {})
graph.add_node("Khatam", Khatam)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", "checkExit")

graph.add_conditional_edges(
    "checkExit",
    checkExit,
    {
        "chat_node": "chat_node",
        "Khatam": "Khatam"
    }
)

graph.add_edge("Khatam", END)

chatbot = graph.compile()

In [34]:
chatbot.invoke({
    "chat_history": []
})

NameError: name 'AIMessage' is not defined